In [0]:
import requests, json
from datetime import datetime
import time


In [0]:
tickers = ["AAPL" , "MSFT", "GOOG", "FGI"]
landing_path = "/Volumes/alpha_vantage/bronze/bronze-landing"

In [0]:
def fetch_raw_time(ticker):
    url = "https://www.alphavantage.co/query"
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "datatype": "json",
        "outputsize": "compact",
        "apikey": "5AJFGTFCW4KW1QUE"
    }
    response = requests.get(url, params=querystring)
    # print(response.json())
    response.raise_for_status()
    return response.json()["Time Series (Daily)"]
def fetch_raw_quote(ticker):
    url = "https://www.alphavantage.co/query"
    querystring = {
        "function": "GLOBAL_QUOTE",
        "symbol": ticker,
        "datatype": "json",
        "outputsize": "compact",
        "apikey": "5AJFGTFCW4KW1QUE"
    }
    response = requests.get(url, params=querystring)
    print(response.json())
    response.raise_for_status()
    
    return response.json()["Global Quote"]

In [0]:
for ticker in tickers:
    time.sleep(15)
    try:
        raw_data = fetch_raw_time(ticker)
        time.sleep(15)
        quote = fetch_raw_quote(ticker)
        raw_data["time_series_daily"] = raw_data.pop("Time Series (Daily)")
        quote["global_quote"] = quote.pop("Global Quote")
        payload = {"symbol": ticker, "timeseries": raw_data, "quote": quote}
        ts = datetime.utcnow().strftime("%Y%m%d%H%M%S")
        out_path = f"{landing_path}/{ticker}/{ticker}_{ts}.json"
        dbutils.fs.put(out_path, json.dumps(payload), overwrite=True)
        print(f"Landed {ticker} -> {out_path}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {ticker}: {e}")
    except (KeyError, ValueError) as e:
        print(f"Unexpected response for {ticker}: {e}")

Unexpected response for AAPL: 'Time Series (Daily)'
Unexpected response for MSFT: 'Time Series (Daily)'
Unexpected response for GOOG: 'Time Series (Daily)'
